# Imports

In [ ]:
import pmdarima as pm
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.set_printoptions(precision=4)

# Load Data

In [ ]:
df = pd.read_csv("data/univariate.csv", header=0, parse_dates=True, index_col=0)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
plt.plot(df)
plt.show()

> Clearly there is no trend.

# Split train and test

In [ ]:
n = int(len(df) * 0.8)

In [ ]:
df_train = df[:n]
df_test = df[n:]

# Auto Arima

In [ ]:
model = pm.auto_arima(y=df_train, seasonal=False, m=0, trace=True)

In [ ]:
model.params()

# Eval

## Train

In [ ]:
pred_train = model.predict_in_sample()

In [ ]:
rmse_train = np.sqrt(mean_squared_error(df_train, pred_train)).item()
rmse_train

In [ ]:
plt.plot(df_train)
plt.plot(pred_train)
plt.plot(df_train[0:1], label = f"Train RMSE: {rmse_train:.4f}")
plt.legend()
plt.show()

## Test

In [ ]:
pred_test = model.predict(n_periods=len(df_test))

In [ ]:
rmse_test = np.sqrt(mean_squared_error(df_test, pred_test)).item()
rmse_test

In [ ]:
plt.plot(df_test)
plt.plot(pred_test)
plt.plot(df_test[0:1], label = f"Test RMSE: {rmse_test:.4f}")
plt.legend()
plt.show()

# Out-of-sample predictions

In [ ]:
model.fit(df['energy_demand'])

In [ ]:
plt.plot(model.predict_in_sample())
plt.plot(pred_train)
plt.plot(pred_test)
plt.show()

In [ ]:
pred_periods = 12

In [ ]:
preds = model.predict(n_periods = pred_periods)

In [ ]:
preds

In [ ]:
plt.plot(df['energy_demand'])
plt.plot(preds)
plt.title(f"Prediction for the next {pred_periods} months")
plt.show()

In [ ]:
# Returning confidence interval

pred_ = model.predict(n_periods = pred_periods, return_conf_int=True)

In [ ]:
type(pred_)

In [ ]:
pred_[0].values

In [ ]:
predictions_df = pd.DataFrame(
    data=np.c_[pred_[0].values, pred_[1][:,0], pred_[1][:,1]],
    columns=['predictions', 'lower_bound', 'upper_bound']
)

In [ ]:
predictions_df

# End